# Surgical Data Cleaning & Column Exploration Playground

This notebook contains the interactive, column-by-column cleaning, standardizing, and engineering steps for the **Multimodal Periodontal Disease** research. 

We go **sheet-by-sheet, column-by-column, and file-by-file** to reach the perfect data state visually before generating the final production `.py` script.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# 1. Ensure project root is in sys.path so we can import src without ModuleNotFoundError
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Configure Pandas display to show ALL columns & rows scrolable without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

print("Environment initialized successfully!")
print("Project root path:", project_root)

Environment initialized successfully!
Project root path: c:\Users\abura\Development\CAF


## Sheet 1: Demographic Info

Below is the surgical cleaning cell for the demographics sheet. It strips trailing spaces, maps redundant education categories, and imputes clean averages.

In [2]:
# =====================================================================
# SHEET 1: DEMOGRAPHICS CLEANING & STANDARDIZATION (LOWERCASE-FIRST)
# =====================================================================

# Load raw demographics sheet
xls_path = os.path.join(project_root, 'data', 'questionnaire.xlsx')
xls = pd.ExcelFile(xls_path)
df_demo = pd.read_excel(xls, sheet_name=0)

# Excluded cohort IDs
drop_ids = {18, 20, 23, 40, 65, 68}

# 1. Clean Patient Name and ID
df_demo = df_demo.rename(columns={df_demo.columns[0]: 'patient_name'})
df_demo = df_demo.dropna(subset=['id'])
df_demo['id'] = df_demo['id'].astype(float).astype(int)
df_demo = df_demo[~df_demo['id'].isin(drop_ids)]

# 2. Drop student id
if 'student id' in df_demo.columns:
    df_demo = df_demo.drop(columns=['student id'])

# 3. STRIP AND CONVERT ALL STRING COLUMNS TO LOWERCASE IMMEDIATELY
for col in df_demo.columns:
    if pd.api.types.is_string_dtype(df_demo[col]) or df_demo[col].dtype == object:
        df_demo[col] = df_demo[col].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)

# 4. Standardize Gender
# (Case conversion automatically merged 'Male' and 'male' into 'male')

# 5. Clean Age
df_demo['Age'] = pd.to_numeric(df_demo['Age'], errors='coerce')
mean_age = df_demo['Age'].mean() # Recalculates to 22.44 after drops
df_demo['Age'] = df_demo['Age'].fillna(mean_age).round().astype(int)

# 6. Clean Education Level (Unify spelling typos and merge duplicate categories in lowercase)
education_mapping = {
    'some college': 'some college',
    'some collage': 'some college',
    'masters degree': "master's degree",
    'bachelors degree': "bachelor's degree",
    "bachelor's degree": "bachelor's degree",
    'some college or associate degree': 'some college or associate degree',
    'high school diploma or associate degree': 'high school diploma or associate degree',
    'high school diploma or equivalent': 'high school diploma or equivalent'
}
df_demo['Highest Education Level Completed'] = df_demo['Highest Education Level Completed'].replace(education_mapping)
df_demo['Highest Education Level Completed'] = df_demo['Highest Education Level Completed'].fillna('some college')

# 7. Clean Occupation
# (Case conversion automatically merged 'Student' and 'student' into 'student')
df_demo['Occupation'] = df_demo['Occupation'].fillna('student')

print("Sheet 1 (Demographics) Cleaning complete!")
print("Active cohort size:", len(df_demo))
print("Unique Genders:", df_demo['Gender'].unique().tolist())
print("Unique Occupations:", df_demo['Occupation'].unique().tolist())
print("Unique Education levels:", df_demo['Highest Education Level Completed'].unique().tolist())

df_demo

Sheet 1 (Demographics) Cleaning complete!
Active cohort size: 65
Unique Genders: ['male', 'female']
Unique Occupations: ['student']
Unique Education levels: ['some college', "master's degree", "bachelor's degree", 'some college or associate degree', 'high school diploma or associate degree', 'high school diploma or equivalent']


,patient_name,id,Gender,Age,Highest Education Level Completed,Occupation
0,ahmad baker,1,male,21,some college,student
1,omar ahmed,2,male,22,some college,student
2,yousef aloudat,3,male,22,some college,student
3,zaid alhawamleh,4,male,24,some college,student
4,sanad alkurdi,5,male,22,some college,student
5,emran bani omar,6,male,23,some college,student
6,abdullah alwawdeh,7,male,21,some college,student
7,mohammed abdelhadi,8,male,22,some college,student
8,odai iyad daghash,9,male,22,some college,student
9,hussen alshrideh,10,male,35,master's degree,student


In [3]:
print(df_demo.isnull().sum())

patient_name                         0
id                                   0
Gender                               0
Age                                  0
Highest Education Level Completed    0
Occupation                           0
dtype: int64


In [4]:
# =====================================================================
# SHEET 2: VAPING HABITS CLEANING & STANDARDIZATION (LOWERCASE-FIRST)
# =====================================================================

# Load raw Vaping Habits sheet (index 1)
df_vape = pd.read_excel(xls, sheet_name=1)

# 1. Standardize column names to lowercase and handle 'id'
df_vape.columns = [c.lower() if c.lower() == 'id' else c for c in df_vape.columns]
df_vape = df_vape.dropna(subset=['id'])
df_vape['id'] = df_vape['id'].astype(float).astype(int)
df_vape = df_vape[~df_vape['id'].isin(drop_ids)]

# 2. STRIP AND CONVERT ALL STRING COLUMNS TO LOWERCASE IMMEDIATELY
for col in df_vape.columns:
    if pd.api.types.is_string_dtype(df_vape[col]) or df_vape[col].dtype == object:
        df_vape[col] = df_vape[col].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)

# 3. Clean 'How often do you vape?'
df_vape['How often do you vape?'] = df_vape['How often do you vape?'].replace({
    'rarely(less than once a week )': 'rarely (less than once a week)',
    'rarely(less than once a week)': 'rarely (less than once a week)',
    'rarely (less than once a week )': 'rarely (less than once a week)'
})
# Non-vapers have NaN. Impute with '0'
df_vape['How often do you vape?'] = df_vape['How often do you vape?'].fillna('0')

# 4. Clean Nicotine Concentration
# Unify: '30 mg /ml' vs '30 mg/ml'
df_vape['What nicotine concentration do you typically use?'] = df_vape['What nicotine concentration do you typically use?'].replace({
    '30 mg /ml (3%)': '30 mg/ml (3%)',
    '30 mg/ml (3%)': '30 mg/ml (3%)'
})
# Non-vapers have NaN. Impute with '0'
df_vape['What nicotine concentration do you typically use?'] = df_vape['What nicotine concentration do you typically use?'].fillna('0')

# 5. Clean Vaping Duration
# Unify: 'less than 6 months ago' into 'less than 6 months'
df_vape['If yes, how long have you been using e-cigarettes?'] = df_vape['If yes, how long have you been using e-cigarettes?'].replace({
    'less than 6 months ago': 'less than 6 months'
})
# Non-vapers have NaN. Impute with '0'
df_vape['If yes, how long have you been using e-cigarettes?'] = df_vape['If yes, how long have you been using e-cigarettes?'].fillna('0')

# 6. Clean Vaping Device
# Non-vapers have NaN. Impute with '0'
df_vape['What type of e-cigarette device do you mainly use?'] = df_vape['What type of e-cigarette device do you mainly use?'].fillna('0')

# 7. Fill generic vape usage questions (yes/no)
df_vape['Have you ever used an e-cigarette (vape)?'] = df_vape['Have you ever used an e-cigarette (vape)?'].fillna('no')
df_vape['Are you currently using e-cigarettes?'] = df_vape['Are you currently using e-cigarettes?'].fillna('no')

# =============================================================================

col_ever = "Have you ever used an e-cigarette (vape)?"
col_current = "Are you currently using e-cigarettes?"
col_duration = "If yes, how long have you been using e-cigarettes?"
col_freq = "How often do you vape?"
col_device = "What type of e-cigarette device do you mainly use?"
col_nic = "What nicotine concentration do you typically use?"
col_smoke_prior = "Have you smoked traditional cigarettes prior to using e-cigarettes?"
col_smoke_dur = "If so, how long did you smoke traditional cigarettes?"

# 1. Standardize column names to lowercase and handle 'id'
df_vape.columns = [c.lower() if c.lower() == 'id' else c for c in df_vape.columns]
df_vape = df_vape.dropna(subset=['id'])
df_vape['id'] = df_vape['id'].astype(float).astype(int)
df_vape = df_vape[~df_vape['id'].isin(drop_ids)]

# 2. STRIP AND CONVERT ALL STRING COLUMNS TO LOWERCASE IMMEDIATELY
for col in df_vape.columns:
    if pd.api.types.is_string_dtype(df_vape[col]) or df_vape[col].dtype == object:
        df_vape[col] = df_vape[col].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)

# 3. Standardize Vaping Frequency
df_vape[col_freq] = df_vape[col_freq].replace({
    'rarely(less than once a week )': 'rarely (less than once a week)',
    'rarely(less than once a week)': 'rarely (less than once a week)',
    'rarely (less than once a week )': 'rarely (less than once a week)'
})

# 4. Standardize Nicotine Concentration
df_vape[col_nic] = df_vape[col_nic].replace({
    '30 mg /ml (3%)': '30 mg/ml (3%)',
    '30 mg/ml (3%)': '30 mg/ml (3%)'
})

# 5. Standardize Vaping Duration
df_vape[col_duration] = df_vape[col_duration].replace({
    'less than 6 months ago': 'less than 6 months'
})

# 6. Standardize Traditional Cigarette Prior Smoking & Duration
df_vape[col_smoke_prior] = df_vape[col_smoke_prior].fillna('no')
df_vape[col_smoke_dur] = df_vape[col_smoke_dur].replace({'never': 'never'}).fillna('never')

# 7. ENFORCE STRICT SURVEY LOGIC ALIGNMENT
# Rule A: If they have never used vapes, force all vaping indicators to '0'
non_vaper_mask = (df_vape[col_ever] == 'no') & (df_vape[col_current] == 'no')
df_vape.loc[non_vaper_mask, col_duration] = '0'
df_vape.loc[non_vaper_mask, col_freq] = '0'
df_vape.loc[non_vaper_mask, col_device] = '0'
df_vape.loc[non_vaper_mask, col_nic] = '0'

# Rule B: If they have never smoked traditional cigarettes prior, force duration to 'never'
non_smoker_mask = (df_vape[col_smoke_prior] == 'no')
df_vape.loc[non_smoker_mask, col_smoke_dur] = 'never'

# 8. Fill any remaining NaNs safely with '0'
df_vape[col_ever] = df_vape[col_ever].fillna('no')
df_vape[col_current] = df_vape[col_current].fillna('no')
df_vape[col_duration] = df_vape[col_duration].fillna('0')
df_vape[col_freq] = df_vape[col_freq].fillna('0')
df_vape[col_device] = df_vape[col_device].fillna('0')
df_vape[col_nic] = df_vape[col_nic].fillna('0')

print("Sheet 2 (Vaping Habits) Cleaning complete!")
print("Unique frequencies:", df_vape[col_freq].unique().tolist())
print("Unique nicotine concentrations:", df_vape[col_nic].unique().tolist())
print("Unique durations:", df_vape[col_duration].unique().tolist())
print("Unique traditional smoking durations:", df_vape[col_smoke_dur].unique().tolist())

# Display sample of cleaned rows to inspect Patient 3 (ID 3) and Patient 6 (ID 6)
df_vape

Sheet 2 (Vaping Habits) Cleaning complete!
Unique frequencies: ['0', 'rarely (less than once a week)', 'constantly throughout the day (more than 5 times per day)']
Unique nicotine concentrations: ['0', '30 mg/ml (3%)', '0 mg/ml (nicotine free)', 'more than 50 mg/ml (5% or higher)', '50 mg/ml (5%)', '20 mg/ml (2%)', '25 mg/ml (2.5%)']
Unique durations: ['0', '6 months to 1 year', '3-5 years', '1-3 years', 'more than 5 years', 'less than 6 months']
Unique traditional smoking durations: ['never', '1 year', '3 years', '4 years', '2 years', '3 months', '5 years', '12 years', '7 years', '6 months']


,id,Have you ever used an e-cigarette (vape)?,Are you currently using e-cigarettes?,"If yes, how long have you been using e-cigarettes?",How often do you vape?,What type of e-cigarette device do you mainly use?,What nicotine concentration do you typically use?,Have you smoked traditional cigarettes prior to using e-cigarettes?,"If so, how long did you smoke traditional cigarettes?"
0,1,no,no,0,0,0,0,no,never
1,2,yes,yes,6 months to 1 year,rarely (less than once a week),pod system,30 mg/ml (3%),no,never
2,3,no,no,0,0,0,0,no,never
3,4,yes,yes,3-5 years,constantly throughout the day (more than 5 times per day),box mod or vape mod,0 mg/ml (nicotine free),no,never
4,5,yes,yes,1-3 years,rarely (less than once a week),pod system,more than 50 mg/ml (5% or higher),yes,1 year
5,6,no,no,0,0,0,0,no,never
6,7,yes,no,0,0,0,0,yes,3 years
7,8,no,no,0,0,0,0,no,never
8,9,yes,yes,3-5 years,constantly throughout the day (more than 5 times per day),disposable vape,50 mg/ml (5%),yes,4 years
9,10,no,no,0,0,0,0,no,never


In [5]:
# =====================================================================
# SHEET 3: ORAL HYGIENE CLEANING & FEATURE ENGINEERING (UPGRADED)
# =====================================================================

# Load raw Oral Hygiene sheet (index 2)
df_hygiene = pd.read_excel(xls, sheet_name=2)

# Define column shortcuts to prevent typing errors
col_brush = "How often do you brush your teeth?"
col_change = "How often do you change your toothbrush?"
col_floss = "Do you use dental floss or interdental brushes as part of your oral care routine?"
col_aids = "Do you prefer using other interdental cleaning aids?"
col_wash = "Do you use mouthwash or rinses on a regular basis?"
col_wash_daily = "Do you use mouthwash daily?"
col_dentist = "How frequently do you visit the dentist for regular check-ups?"
col_cleaning = "How often do you go for professional dental cleanings?"
col_recent = "When was your most recent professional dental cleaning?"
col_ortho = "Have you ever undergone orthodontic treatment?"
col_gum_disease = "Have you ever been diagnosed with gum disease (gingivitis or periodontitis)?"
col_perio_tx = "Have you ever received periodontal treatment?"
col_bleach = "Have you ever undergone a tooth bleaching procedure?"

# 1. Standardize column names to lowercase and handle 'id'
df_hygiene.columns = [c.lower() if c.lower() == 'id' else c for c in df_hygiene.columns]
df_hygiene = df_hygiene.dropna(subset=['id'])
df_hygiene['id'] = df_hygiene['id'].astype(float).astype(int)
df_hygiene = df_hygiene[~df_hygiene['id'].isin(drop_ids)]

# 2. STRIP AND CONVERT ALL STRING COLUMNS TO LOWERCASE IMMEDIATELY
for col in df_hygiene.columns:
    if pd.api.types.is_string_dtype(df_hygiene[col]) or df_hygiene[col].dtype == object:
        df_hygiene[col] = df_hygiene[col].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)

# 3. Clean Teeth Brushing Frequency (Fix typos: 'rarley' -> 'rarely', 'occasionaly' -> 'occasionally')
df_hygiene[col_brush] = df_hygiene[col_brush].replace({
    'rarley': 'rarely',
    'occasionaly': 'occasionally'
}).fillna('once a day')

# 4. Clean Toothbrush Change Frequency (Unify all variations)
df_hygiene[col_change] = df_hygiene[col_change].replace({
    'every 3 to 4 months': 'every 3-4 months',
    'other: i dont change': 'never',
    'other:no': 'never',
    'other ( never )': 'never',
    'other:(7 months)': 'every 6 months or more',
    'every 6 months': 'every 6 months or more'
}).fillna('every 3-4 months')

# 5. Clean Dental Floss / Interdental brushes
df_hygiene[col_floss] = df_hygiene[col_floss].fillna('no')

# 6. CATEGORY D: EXTRACT MULTI-HOT BINARY COLUMNS FROM INTERDENTAL AIDS
df_hygiene['uses_miswak'] = df_hygiene[col_aids].apply(lambda x: 1 if isinstance(x, str) and 'miswak' in x else 0)
df_hygiene['uses_tongue_scraper'] = df_hygiene[col_aids].apply(lambda x: 1 if isinstance(x, str) and 'tongue' in x else 0)
df_hygiene['uses_interdental_brushes'] = df_hygiene[col_aids].apply(lambda x: 1 if isinstance(x, str) and 'interdental' in x else 0)
df_hygiene['uses_floss_pick'] = df_hygiene[col_aids].apply(lambda x: 1 if isinstance(x, str) and 'floss pick' in x else 0)
df_hygiene['uses_water_irrigator'] = df_hygiene[col_aids].apply(lambda x: 1 if isinstance(x, str) and 'irrigator' in x else 0)
df_hygiene['uses_toothpick'] = df_hygiene[col_aids].apply(lambda x: 1 if isinstance(x, str) and 'toothpick' in x else 0)

# Replace original text aids column with standard yes/no indicator
df_hygiene[col_aids] = df_hygiene[col_aids].apply(lambda x: 'no' if pd.isna(x) or x == 'no' else 'yes')

# 7. Clean Mouthwash questions
df_hygiene[col_wash] = df_hygiene[col_wash].fillna('no')
df_hygiene[col_wash_daily] = df_hygiene[col_wash_daily].fillna('no')

# 8. Clean Dentist Visits (Fix typos)
df_hygiene[col_dentist] = df_hygiene[col_dentist].replace({
    'evey 3-4 months': 'every 3-4 months'
}).fillna('only when needed')

# 9. Clean Dental Cleanings Frequency (Fix recency answer 'less than 6 months ago' -> 'every 6 months')
df_hygiene[col_cleaning] = df_hygiene[col_cleaning].replace({
    'evey 3-4 months': 'every 3-4 months',
    'less than 6 months ago': 'every 6 months'
}).fillna('only when needed')

# 10. Clean Most Recent Cleaning (Fix typos)
df_hygiene[col_recent] = df_hygiene[col_recent].replace({
    'aout a year ago': 'about a year ago'
}).fillna('not sure')

# 11. Clean Ortho, Gum Disease, Perio treatments
df_hygiene[col_ortho] = df_hygiene[col_ortho].fillna('no')
df_hygiene[col_gum_disease] = df_hygiene[col_gum_disease].fillna('not sure')
df_hygiene[col_perio_tx] = df_hygiene[col_perio_tx].fillna('no')

# 12. Clean Tooth Bleaching (Simplify and map)
df_hygiene[col_bleach] = df_hygiene[col_bleach].replace({
    'other (never received tooth bleaching)': 'never',
    'other ( never received tooth bleaching)': 'never',
    'home bleaching (using tooth splints)': 'home bleaching',
    'in office bleaching (vital tooth splints)': 'in office bleaching',
    'intracoronal bleaching( nonvital bleaching of a single tooth)': 'intracoronal bleaching'
}).fillna('never')

print("Sheet 3 (Oral Hygiene) Cleaning & Feature Engineering complete!")
print("Unique Brushing Freqs:", df_hygiene[col_brush].unique().tolist())
print("Unique Toothbrush Changes:", df_hygiene[col_change].unique().tolist())
print("Unique dentist visits:", df_hygiene[col_dentist].unique().tolist())
print("Unique cleanings:", df_hygiene[col_cleaning].unique().tolist())
print("Unique Bleaching types:", df_hygiene[col_bleach].unique().tolist())

print("\n--- Diagnostic check on Multi-Hot Interdental Aids ---")
print("Total patients using Miswak:", df_hygiene['uses_miswak'].sum())
print("Total patients using Tongue Scraper:", df_hygiene['uses_tongue_scraper'].sum())
print("Total patients using Interdental Brushes (Excluding ID 23):", df_hygiene['uses_interdental_brushes'].sum())
print("Total patients using Floss Pick:", df_hygiene['uses_floss_pick'].sum())
print("Total patients using Water Irrigator:", df_hygiene['uses_water_irrigator'].sum())
print("Total patients using Toothpick:", df_hygiene['uses_toothpick'].sum())

print("\n--- Columns in the Cleaned Dataframe ---")
print(df_hygiene.columns.tolist())

# Display sample of rows
df_hygiene.head(10)

Sheet 3 (Oral Hygiene) Cleaning & Feature Engineering complete!
Unique Brushing Freqs: ['twice a day', 'once a day', 'rarely', 'occasionally', 'more than twice a day']
Unique Toothbrush Changes: ['every 3-4 months', 'every couple of months', 'never', 'once a month', 'every 6 months or more']
Unique dentist visits: ['every 6 months', 'never', 'every 3-4 months', 'only when needed', 'once a year']
Unique cleanings: ['every 6 months', 'only when needed', 'every 3-4 months', 'never', 'once a year']
Unique Bleaching types: ['never', 'home bleaching', 'in office bleaching', 'intracoronal bleaching']

--- Diagnostic check on Multi-Hot Interdental Aids ---
Total patients using Miswak: 1
Total patients using Tongue Scraper: 2
Total patients using Interdental Brushes (Excluding ID 23): 0
Total patients using Floss Pick: 1
Total patients using Water Irrigator: 1
Total patients using Toothpick: 3

--- Columns in the Cleaned Dataframe ---
['id', 'How often do you brush your teeth?', 'How often do y

,id,How often do you brush your teeth?,How often do you change your toothbrush?,Do you use dental floss or interdental brushes as part of your oral care routine?,Do you prefer using other interdental cleaning aids?,Do you use mouthwash or rinses on a regular basis?,Do you use mouthwash daily?,How frequently do you visit the dentist for regular check-ups?,How often do you go for professional dental cleanings?,When was your most recent professional dental cleaning?,Have you ever undergone orthodontic treatment?,Have you ever been diagnosed with gum disease (gingivitis or periodontitis)?,Have you ever received periodontal treatment?,Have you ever undergone a tooth bleaching procedure?,uses_miswak,uses_tongue_scraper,uses_interdental_brushes,uses_floss_pick,uses_water_irrigator,uses_toothpick
0,1,twice a day,every 3-4 months,no,no,yes,yes,every 6 months,every 6 months,less than 6 months ago,no,not sure,yes,never,0,0,0,0,0,0
1,2,twice a day,every couple of months,no,no,no,no,every 6 months,every 6 months,less than 3 months ago,no,no,no,never,0,0,0,0,0,0
2,3,once a day,every 3-4 months,no,no,no,no,never,every 6 months,less than 6 months ago,no,no,no,never,0,0,0,0,0,0
3,4,twice a day,every 3-4 months,yes,no,no,no,every 3-4 months,only when needed,less than 3 months ago,no,not sure,no,never,0,0,0,0,0,0
4,5,twice a day,every couple of months,no,no,yes,yes,every 6 months,every 6 months,less than 3 months ago,no,yes,yes,never,0,0,0,0,0,0
5,6,once a day,every 3-4 months,no,no,no,no,only when needed,every 6 months,less than 6 months ago,no,no,no,never,0,0,0,0,0,0
6,7,once a day,every couple of months,yes,no,no,no,every 6 months,every 3-4 months,less than 6 months ago,no,no,no,never,0,0,0,0,0,0
7,8,twice a day,every 3-4 months,no,no,yes,no,every 3-4 months,never,not sure,no,not sure,no,never,0,0,0,0,0,0
8,9,once a day,every couple of months,no,no,yes,no,once a year,every 6 months,about a month ago,no,no,no,never,0,0,0,0,0,0
9,10,once a day,every 3-4 months,no,no,no,no,only when needed,every 6 months,less than 6 months ago,no,not sure,no,never,0,0,0,0,0,0


In [6]:
# =====================================================================
# SHEET 4: LIFESTYLE FACTORS CLEANING & FEATURE ENGINEERING
# =====================================================================

# Load raw Lifestyle Factors sheet (index 3)
df_lifestyle = pd.read_excel(xls, sheet_name=3)

# Define column shortcuts to prevent spelling mismatches
col_alcohol = "Do you consume alcohol?"
col_alc_freq = "If yes, how often do you consume alcohol?"
col_diet = "Are you currently following any specific diet?"
col_supps = "Are you taking any nutritional supplements or vitamins?"
col_chronic = "Do you have any chronic diseases (e.g., diabetes, cardiovascular disease)?"
col_meds = "Are you currently taking any medications (Cardiovascular drugs, Respiratory drugs, Gastrointestinal drugs, Endocrine drugs)?"
col_med_status = "If so, the medication's prescription status?"
col_med_use = "The medications Based on Therapeutic Use?"
col_med_type = "Oral medications, Topical medications, Inhalation medications, Injectable medications? Please specify:"
col_exercise = "Do you exercise regularly (at least three times a week)?"
col_workout = "Do you work out daily?"

# 1. Standardize column names to lowercase and handle 'id'
df_lifestyle.columns = [c.lower() if c.lower() == 'id' else c for c in df_lifestyle.columns]
df_lifestyle = df_lifestyle.dropna(subset=['id'])
df_lifestyle['id'] = df_lifestyle['id'].astype(float).astype(int)
df_lifestyle = df_lifestyle[~df_lifestyle['id'].isin(drop_ids)]

# 2. STRIP AND CONVERT ALL STRING COLUMNS TO LOWERCASE IMMEDIATELY
for col in df_lifestyle.columns:
    if pd.api.types.is_string_dtype(df_lifestyle[col]) or df_lifestyle[col].dtype == object:
        df_lifestyle[col] = df_lifestyle[col].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)

# 3. Clean Alcohol consumption
df_lifestyle[col_alcohol] = df_lifestyle[col_alcohol].fillna('no')
df_lifestyle[col_alc_freq] = df_lifestyle[col_alc_freq].fillna('0')

# Enforce logic: If alcohol == no, then frequency is '0'
df_lifestyle.loc[df_lifestyle[col_alcohol] == 'no', col_alc_freq] = '0'

# 4. Clean and Standardize specific Diets (Fix typos: 'hogh' -> 'high', 'suger' -> 'sugar')
df_lifestyle[col_diet] = df_lifestyle[col_diet].replace({
    'no special diet': 'no special diet',
    'low sugar diet': 'low sugar diet',
    'other:( low sugar )': 'low sugar diet',
    'high sugar diet': 'high sugar diet',
    'high-suger diet': 'high sugar diet',
    'other:( hogh protein-low sugar)': 'high protein low sugar',
    'other: (keto)': 'keto',
    'other: ( low carbs)': 'low carbs'
}).fillna('no special diet')

# 5. CATEGORY D: EXTRACT MULTI-HOT BINARY COLUMNS FROM NUTRIENTS/SUPPLEMENTS
df_lifestyle['takes_creatine'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['creatine', 'creatin', 'ceriatine']) else 0
)
df_lifestyle['takes_omega3'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and 'omega' in x else 0
)
df_lifestyle['takes_vitamin_d'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['vit d', 'vitamin d', 'd3', ' b12 and d']) else 0
)
df_lifestyle['takes_zinc'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['zinc', 'zink']) else 0
)
df_lifestyle['takes_multivitamins'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['multi', 'multivitamins']) else 0
)
df_lifestyle['takes_magnesium'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['magnesium', 'magneisum']) else 0
)
df_lifestyle['takes_protein'] = df_lifestyle[col_supps].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['protein', 'protien']) else 0
)

# Standardize the original supplements column to yes/no
df_lifestyle[col_supps] = df_lifestyle[col_supps].apply(lambda x: 'no' if pd.isna(x) or x == 'no' else 'yes')

# 6. CATEGORY D: EXTRACT MULTI-HOT BINARY COLUMNS FROM CHRONIC DISEASES
df_lifestyle['has_asthma'] = df_lifestyle[col_chronic].apply(
    lambda x: 1 if isinstance(x, str) and 'asthma' in x else 0
)
df_lifestyle['has_lupus'] = df_lifestyle[col_chronic].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['lupus', 'sle']) else 0
)
df_lifestyle['has_psoriasis'] = df_lifestyle[col_chronic].apply(
    lambda x: 1 if isinstance(x, str) and any(kw in x for kw in ['psoriasis', 'psoanisis']) else 0
)
df_lifestyle['has_hypothyroidism'] = df_lifestyle[col_chronic].apply(
    lambda x: 1 if isinstance(x, str) and 'hypothyroid' in x else 0
)

# Standardize the original chronic diseases column to yes/no
df_lifestyle[col_chronic] = df_lifestyle[col_chronic].apply(lambda x: 'no' if pd.isna(x) or x == 'no' else 'yes')

# 7. Clean and Standardize Medications
df_lifestyle[col_meds] = df_lifestyle[col_meds].fillna('no')
df_lifestyle[col_med_status] = df_lifestyle[col_med_status].replace({
    'prescription medications': 'prescription medications'
}).fillna('none')
df_lifestyle[col_med_use] = df_lifestyle[col_med_use].replace({
    'antihistamne': 'antihistamine'
}).fillna('none')
df_lifestyle[col_med_type] = df_lifestyle[col_med_type].fillna('none')

# Enforce logic: If taking medications == no, force follow-ups to 'none'
df_lifestyle.loc[df_lifestyle[col_meds] == 'no', [col_med_status, col_med_use, col_med_type]] = 'none'

# 8. Clean Exercise & Workouts (Fix spacing variations)
df_lifestyle[col_exercise] = df_lifestyle[col_exercise].fillna('no')
df_lifestyle[col_workout] = df_lifestyle[col_workout].replace({
    'i used to exercise in the past': 'used to exercise in the past'
}).fillna('never')

print("Sheet 4 (Lifestyle Factors) Cleaning & Feature Engineering complete!")
print("Unique alcohol frequencies:", df_lifestyle[col_alc_freq].unique().tolist())
print("Unique Diets:", df_lifestyle[col_diet].unique().tolist())
print("Unique Chronic Diseases:", df_lifestyle[col_chronic].unique().tolist())
print("Unique workouts:", df_lifestyle[col_workout].unique().tolist())

print("\n--- Diagnostic check on Supplements Multi-Hot ---")
print("Takes Creatine:", df_lifestyle['takes_creatine'].sum())
print("Takes Omega-3:", df_lifestyle['takes_omega3'].sum())
print("Takes Vitamin D:", df_lifestyle['takes_vitamin_d'].sum())
print("Takes Zinc:", df_lifestyle['takes_zinc'].sum())
print("Takes Multivitamins:", df_lifestyle['takes_multivitamins'].sum())
print("Takes Magnesium:", df_lifestyle['takes_magnesium'].sum())
print("Takes Protein:", df_lifestyle['takes_protein'].sum())

print("\n--- Diagnostic check on Chronic Diseases Multi-Hot ---")
print("Has Asthma:", df_lifestyle['has_asthma'].sum())
print("Has Lupus (SLE):", df_lifestyle['has_lupus'].sum())
print("Has Psoriasis:", df_lifestyle['has_psoriasis'].sum())
print("Has Hypothyroidism:", df_lifestyle['has_hypothyroidism'].sum())

print("\n--- Columns in the Cleaned Dataframe ---")
print(df_lifestyle.columns.tolist())

# Display sample of rows
df_lifestyle.head(6)

Sheet 4 (Lifestyle Factors) Cleaning & Feature Engineering complete!
Unique alcohol frequencies: ['0', 'rarely ( less than once a month)', 'occasionally (1-3 times a month)']
Unique Diets: ['no special diet', 'low sugar diet', 'high protein low sugar', 'high sugar diet', 'keto', 'low carbs', 'other:(low sugar)']
Unique Chronic Diseases: ['no', 'yes']
Unique workouts: ['never', 'yes', 'used to exercise in the past', 'no']

--- Diagnostic check on Supplements Multi-Hot ---
Takes Creatine: 9
Takes Omega-3: 11
Takes Vitamin D: 12
Takes Zinc: 5
Takes Multivitamins: 9
Takes Magnesium: 4
Takes Protein: 3

--- Diagnostic check on Chronic Diseases Multi-Hot ---
Has Asthma: 2
Has Lupus (SLE): 1
Has Psoriasis: 1
Has Hypothyroidism: 1

--- Columns in the Cleaned Dataframe ---
['id', 'Do you consume alcohol?', 'If yes, how often do you consume alcohol?', 'Are you currently following any specific diet?', 'Are you taking any nutritional supplements or vitamins?', 'Do you have any chronic diseases (e.

,id,Do you consume alcohol?,"If yes, how often do you consume alcohol?",Are you currently following any specific diet?,Are you taking any nutritional supplements or vitamins?,"Do you have any chronic diseases (e.g., diabetes, cardiovascular disease)?","Are you currently taking any medications (Cardiovascular drugs, Respiratory drugs, Gastrointestinal drugs, Endocrine drugs)?","If so, the medication's prescription status?",The medications Based on Therapeutic Use?,"Oral medications, Topical medications, Inhalation medications, Injectable medications? Please specify:",Do you exercise regularly (at least three times a week)?,Do you work out daily?,takes_creatine,takes_omega3,takes_vitamin_d,takes_zinc,takes_multivitamins,takes_magnesium,takes_protein,has_asthma,has_lupus,has_psoriasis,has_hypothyroidism
0,1,no,0,no special diet,yes,no,no,none,none,none,yes,never,0,1,0,0,1,0,0,0,0,0,0
1,2,no,0,no special diet,no,no,no,none,none,none,yes,yes,0,0,0,0,0,0,0,0,0,0,0
2,3,no,0,no special diet,no,no,no,none,none,none,no,used to exercise in the past,0,0,0,0,0,0,0,0,0,0,0
3,4,no,0,no special diet,no,no,no,none,none,none,no,no,0,0,0,0,0,0,0,0,0,0,0
4,5,no,0,no special diet,no,no,no,none,none,none,no,no,0,0,0,0,0,0,0,0,0,0,0
5,6,no,0,no special diet,yes,no,no,none,none,none,yes,used to exercise in the past,0,0,0,0,1,0,0,0,0,0,0


In [7]:
# =====================================================================
# SHEET 5: SELF-REPORTED SYMPTOMS CLEANING & STANDARDIZATION
# =====================================================================

# Load raw Symptoms sheet (index 4)
df_symptoms = pd.read_excel(xls, sheet_name=4)

# Define column shortcuts to prevent spelling mismatches
col_bleed = "Do your gums bleed when you brush or floss?"
col_swell = "Have you observed any recent swelling or redness in your gums?"
col_discomfort = "Have you felt any discomfort in your mouth recently?"
col_burning = "Have you recently experienced any burning sensation or tenderness in your oral cavity?"
col_nose = "Do you experience any difficulty breathing through your nose?"
col_loose = "Have you noticed any loose teeth recently?"
col_breath = "Have you noticed bad breath even after brushing?"
col_bruxism = "Have you ever experienced parafunctional habits such as bruxism, clenching, or grinding?"
col_discolor = "Have you noticed tooth discoloration recently?"
col_sensitive = "Have you noticed sensitivity in your teeth or gums recently?"

# 1. Standardize column names to lowercase and handle 'id'
df_symptoms.columns = [c.lower() if c.lower() == 'id' else c for c in df_symptoms.columns]
df_symptoms = df_symptoms.dropna(subset=['id'])
df_symptoms['id'] = df_symptoms['id'].astype(float).astype(int)
df_symptoms = df_symptoms[~df_symptoms['id'].isin(drop_ids)]

# 2. STRIP AND CONVERT ALL STRING COLUMNS TO LOWERCASE IMMEDIATELY
for col in df_symptoms.columns:
    if pd.api.types.is_string_dtype(df_symptoms[col]) or df_symptoms[col].dtype == object:
        df_symptoms[col] = df_symptoms[col].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)

# 3. Clean Bleeding (Standardize: 'often', 'never', 'occasionally')
df_symptoms[col_bleed] = df_symptoms[col_bleed].fillna('never')

# 4. Clean Swelling or redness
df_symptoms[col_swell] = df_symptoms[col_swell].fillna('no')

# 5. Clean Mouth discomfort (Standardize: 'yes', 'no', 'not sure')
df_symptoms[col_discomfort] = df_symptoms[col_discomfort].fillna('no')

# 6. Clean Burning sensation
df_symptoms[col_burning] = df_symptoms[col_burning].fillna('no')

# 7. Clean Nose breathing difficulty
df_symptoms[col_nose] = df_symptoms[col_nose].fillna('no')

# 8. Clean Loose teeth
df_symptoms[col_loose] = df_symptoms[col_loose].fillna('no')

# 9. Clean Bad breath
df_symptoms[col_breath] = df_symptoms[col_breath].fillna('no')

# 10. Clean Bruxism/grinding habits
df_symptoms[col_bruxism] = df_symptoms[col_bruxism].fillna('no')

# 11. Clean Discoloration
df_symptoms[col_discolor] = df_symptoms[col_discolor].fillna('no')

# 12. Clean Sensitivity
df_symptoms[col_sensitive] = df_symptoms[col_sensitive].fillna('no')

print("Sheet 5 (Self-Reported Symptoms) Cleaning complete!")
print("Unique Gum Bleeding Frequencies:", df_symptoms[col_bleed].unique().tolist())
print("Unique Swelling states:", df_symptoms[col_swell].unique().tolist())
print("Unique Discomfort states:", df_symptoms[col_discomfort].unique().tolist())
print("Unique Burning sensation states:", df_symptoms[col_burning].unique().tolist())
print("Unique Bruxism states:", df_symptoms[col_bruxism].unique().tolist())
print("Unique Sensitivity states:", df_symptoms[col_sensitive].unique().tolist())

print("\n--- Columns in the Cleaned Dataframe ---")
print(df_symptoms.columns.tolist())

# Display sample of rows
df_symptoms.head(6)

Sheet 5 (Self-Reported Symptoms) Cleaning complete!
Unique Gum Bleeding Frequencies: ['often', 'never', 'occasionally']
Unique Swelling states: ['yes', 'no']
Unique Discomfort states: ['no', 'yes', 'not sure']
Unique Burning sensation states: ['not sure', 'no', 'yes']
Unique Bruxism states: ['yes', 'no']
Unique Sensitivity states: ['yes', 'no']

--- Columns in the Cleaned Dataframe ---
['id', 'Do your gums bleed when you brush or floss?', 'Have you observed any recent swelling or redness in your gums?', 'Have you felt any discomfort in your mouth recently?', 'Have you recently experienced any burning sensation or tenderness in your oral cavity?', 'Do you experience any difficulty breathing through your nose?', 'Have you noticed any loose teeth recently?', 'Have you noticed bad breath even after brushing?', 'Have you ever experienced parafunctional habits such as bruxism, clenching, or grinding?', 'Have you noticed tooth discoloration recently?', 'Have you noticed sensitivity in your te

,id,Do your gums bleed when you brush or floss?,Have you observed any recent swelling or redness in your gums?,Have you felt any discomfort in your mouth recently?,Have you recently experienced any burning sensation or tenderness in your oral cavity?,Do you experience any difficulty breathing through your nose?,Have you noticed any loose teeth recently?,Have you noticed bad breath even after brushing?,"Have you ever experienced parafunctional habits such as bruxism, clenching, or grinding?",Have you noticed tooth discoloration recently?,Have you noticed sensitivity in your teeth or gums recently?
0,1,often,yes,no,not sure,no,no,yes,yes,no,yes
1,2,never,no,no,no,no,no,no,no,not sure,no
2,3,occasionally,no,no,no,no,no,no,no,yes,yes
3,4,often,yes,no,yes,no,no,no,no,yes,no
4,5,occasionally,yes,yes,yes,yes,yes,yes,yes,no,yes
5,6,often,no,no,no,no,no,no,yes,yes,no


In [8]:
# =====================================================================
# SHEET 6: PERIODONTOGRAM WIDE PIVOT & THE GRAND MASTER MERGE
# =====================================================================

import os
import pandas as pd
import numpy as np

# 1. Load the raw periodontogram dataset
p_path = os.path.join(project_root, 'data', 'periodontogram_dataset.xlsx')
df_p_raw = pd.read_excel(p_path)

# Standardize columns to lowercase
df_p_raw.columns = [c.lower().strip() for c in df_p_raw.columns]
# Standardize Patient ID column
df_p_raw = df_p_raw.rename(columns={'patient_identifier': 'id'})
df_p_raw['id'] = df_p_raw['id'].astype(float).astype(int)

# Exclude invalid cohort IDs
df_p_raw = df_p_raw[~df_p_raw['id'].isin(drop_ids)]

# 2. Clean Surface string representation
df_p_raw['surface'] = df_p_raw['surface'].astype(str).str.strip().str.lower()

# 3. Parse continuous full-mouth clinical indices (BOP & FMPI)
# Strip '%' and scale to a safe float between 0.0 and 1.0 for the AI model
def parse_percentage(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).replace('%', '').strip()
    try:
        return float(val_str) / 100.0
    except ValueError:
        return 0.0

df_p_raw['bop_scaled'] = df_p_raw['bop'].apply(parse_percentage)
df_p_raw['fmpi_scaled'] = df_p_raw['fmpi'].apply(parse_percentage)

# 4. Clean and parse pocket depth Scores
# Strip the quote and split the 3-character string into 3 separate digit integers
def parse_pocket_score(val):
    if pd.isna(val):
        return 0, 0, 0
    # Strip quotes, spaces
    val_str = str(val).replace("'", "").strip()
    # Pad to 3 characters in case of entry truncation (e.g. '10' -> '010')
    if len(val_str) < 3:
        val_str = val_str.zfill(3)
    # Extract mesial, mid, and distal pocket scores
    try:
        s1 = int(val_str[0])
        s2 = int(val_str[1])
        s3 = int(val_str[2])
        return s1, s2, s3
    except (ValueError, IndexError):
        return 0, 0, 0

# 5. PERFORM THE WIDE PIVOT (1 Row per Patient)
# Define standard 28 teeth (excluding wisdom teeth 18, 28, 38, 48) and 2 surfaces
all_teeth = [17, 16, 15, 14, 13, 12, 11, 21, 22, 23, 24, 25, 26, 27, 47, 46, 45, 44, 43, 42, 41, 31, 32, 33, 34, 35, 36, 37]
all_surfaces = ['front', 'back']

pivoted_patients = []
active_patient_ids = sorted(df_p_raw['id'].unique())

for pid in active_patient_ids:
    patient_df = df_p_raw[df_p_raw['id'] == pid]
    
    # Take patient-level indices from the first record
    first_row = patient_df.iloc[0]
    patient_record = {
        'id': pid,
        'bop_scaled': first_row['bop_scaled'],
        'fmpi_scaled': first_row['fmpi_scaled']
    }
    
    # Iterate over all possible teeth and surfaces to create wide columns
    for tooth in all_teeth:
        for surf in all_surfaces:
            # Find the specific row for this tooth and surface
            tooth_row = patient_df[(patient_df['tooth_id'] == tooth) & (patient_df['surface'] == surf)]
            
            col_s1 = f'tooth_{tooth}_{surf}_score_s1'
            col_s2 = f'tooth_{tooth}_{surf}_score_s2'
            col_s3 = f'tooth_{tooth}_{surf}_score_s3'
            col_bleed = f'tooth_{tooth}_{surf}_bleeding'
            col_plaque = f'tooth_{tooth}_{surf}_plaque'
            
            if not tooth_row.empty:
                t_row = tooth_row.iloc[0]
                # Parse pocket score
                s1, s2, s3 = parse_pocket_score(t_row['score'])
                
                patient_record[col_s1] = s1
                patient_record[col_s2] = s2
                patient_record[col_s3] = s3
                patient_record[col_bleed] = int(t_row['bleeding']) if not pd.isna(t_row['bleeding']) else 0
                patient_record[col_plaque] = int(t_row['plaque']) if not pd.isna(t_row['plaque']) else 0
            else:
                # Default empty/missing tooth sites to healthy (0)
                patient_record[col_s1] = 0
                patient_record[col_s2] = 0
                patient_record[col_s3] = 0
                patient_record[col_bleed] = 0
                patient_record[col_plaque] = 0
                
    pivoted_patients.append(patient_record)

df_periodonto = pd.DataFrame(pivoted_patients)

print("Sheet 6 (Periodontogram) Wide-Pivot complete!")
print("Pivoted DataFrame Shape:", df_periodonto.shape)
print("Patient-level metrics check for Patient 1:")
print(f"  BOP Scaled: {df_periodonto.loc[df_periodonto['id'] == 1, 'bop_scaled'].values[0]:.4f}")
print(f"  FMPI Scaled: {df_periodonto.loc[df_periodonto['id'] == 1, 'fmpi_scaled'].values[0]:.4f}")
print("Pocket Depth Site splits check for Patient 1 (Tooth 17 Front):")
print(f"  Site 1: {df_periodonto.loc[df_periodonto['id'] == 1, 'tooth_17_front_score_s1'].values[0]}")
print(f"  Site 2: {df_periodonto.loc[df_periodonto['id'] == 1, 'tooth_17_front_score_s2'].values[0]}")
print(f"  Site 3: {df_periodonto.loc[df_periodonto['id'] == 1, 'tooth_17_front_score_s3'].values[0]}")

# =====================================================================
# THE GRAND MASTER JOIN (Merging Sheets 1, 2, 3, 4, 5, and 6)
# =====================================================================
print("\n--- Starting Grand Master Merge ---")

# Step-by-step merge to prevent ID mismatches
df_master = df_demo.copy()
df_master = pd.merge(df_master, df_vape, on='id', how='inner', suffixes=('', '_vape'))
df_master = pd.merge(df_master, df_hygiene, on='id', how='inner', suffixes=('', '_hygiene'))
df_master = pd.merge(df_master, df_lifestyle, on='id', how='inner', suffixes=('', '_lifestyle'))
df_master = pd.merge(df_master, df_symptoms, on='id', how='inner', suffixes=('', '_symptoms'))
df_master = pd.merge(df_master, df_periodonto, on='id', how='inner', suffixes=('', '_periodonto'))

Sheet 6 (Periodontogram) Wide-Pivot complete!
Pivoted DataFrame Shape: (65, 283)
Patient-level metrics check for Patient 1:
  BOP Scaled: 0.1828
  FMPI Scaled: 0.3710
Pocket Depth Site splits check for Patient 1 (Tooth 17 Front):
  Site 1: 2
  Site 2: 2
  Site 3: 3

--- Starting Grand Master Merge ---


In [9]:
# =====================================================================
# MERGE WITH PARTICIPANT ID (Target Labels & Clinician Ratings)
# =====================================================================

import os
import pandas as pd
import numpy as np

# 1. Load raw Participant ID sheet
participant_path = os.path.join(project_root, 'data', 'PARTICIPANT ID.xlsx')
df_participant = pd.read_excel(participant_path)

# 2. Keep only the relevant columns and drop rows without an ID
df_participant = df_participant.dropna(subset=['id'])
df_participant['id'] = df_participant['id'].astype(float).astype(int)

# Normalize column names of interest
df_participant = df_participant.rename(columns={
    '1-10 pic': '1-10 pic',
    'Gingivitis grading ': 'gingivitis_grading'
})

# 3. Standardize target labels
df_participant['gingivitis_grading'] = (
    df_participant['gingivitis_grading']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Map grading categories to binary outcomes
label_map = {
    'grade 0': 0, 'garde 0': 0, 'grade o': 0, 'grdae 0': 0,
    'grade 1': 1, 'garde 1': 1,
    'grade 2': 1
}
df_participant['label'] = df_participant['gingivitis_grading'].map(label_map)

# 4. Strictly drop any rows with missing labels OR missing clinical picture scores
df_participant = df_participant.dropna(subset=['label', '1-10 pic'])
df_participant['label'] = df_participant['label'].astype(int)
df_participant['1-10 pic'] = df_participant['1-10 pic'].astype(float)

# 5. Inner Join with Cleaned Master Dataframe (only keeping patients with complete questionnaire AND clinician data)
df_final = pd.merge(
    df_master, 
    df_participant[['id', '1-10 pic', 'label']], 
    on='id', 
    how='inner'
)

# 6. Save final cleaned dataset
final_xlsx_path = os.path.join(project_root, 'data', 'questionnaire_cleaned.xlsx')
df_final.to_excel(final_xlsx_path, index=False)

print("FINAL DATASET GENERATION SUCCESSFUL")
print("Final Dataset Shape:", df_final.shape)
print("Label Distribution:\n", df_final['label'].value_counts())
print(f"Final dataset saved to: {final_xlsx_path}")


FINAL DATASET GENERATION SUCCESSFUL
Final Dataset Shape: (50, 349)
Label Distribution:
 label
1    25
0    25
Name: count, dtype: int64
Final dataset saved to: c:\Users\abura\Development\CAF\data\questionnaire_cleaned.xlsx
